In [ ]:
# import torch
# import gc

# # Run garbage collector
# gc.collect()

# # Empty PyTorch CUDA cache
# torch.cuda.empty_cache()

# # Forcibly clear memory held by caching allocator
# torch.cuda.ipc_collect()

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from transformers import DataCollatorForLanguageModeling
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model
from datasets import load_dataset
from transformers import BitsAndBytesConfig
import transformers

transformers.logging.set_verbosity_info()

model_id = "gpt2-medium"
output_dir = "models/gpt2-medium-it-alpaca"

# 1. Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token

# 2. Quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype="bfloat16"  # or torch.float16 if bfloat16 unsupported,
)

# 3. Load quantized model
from accelerate import Accelerator
accelerator = Accelerator()

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map={"": accelerator.process_index}
)
model = accelerator.prepare(model)

# 4. Enable LoRA
model = prepare_model_for_kbit_training(model)
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["c_attn", "c_proj"]  # ✅ Required for GPT-2
)
model = get_peft_model(model, peft_config)

# 5. Dataset
dataset = load_dataset("tatsu-lab/alpaca", split="train[:5000]")  # very small subset
split_dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split_dataset["train"]
test_dataset = split_dataset["test"]

def tokenize(example):
    return tokenizer(example["text"], truncation=True, padding="max_length", max_length=1024)

tokenized_train = train_dataset.map(tokenize, batched=True)
tokenized_test = test_dataset.map(tokenize, batched=True)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# 6. Training arguments
training_args = TrainingArguments(
    per_device_train_batch_size=4,
    gradient_accumulation_steps=16,
    learning_rate=6e-4,
    num_train_epochs=5,
    logging_steps=100,
    output_dir=output_dir,
    save_strategy="epoch",
    fp16=True,  # use bf16=True if available
    report_to="none",
    ddp_find_unused_parameters=False  # ✅ important for LoRA multi-GPU
)

# 7. Train
print("Training on devices:", model.hf_device_map)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    data_collator=data_collator
)
trainer.train()


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel, PeftConfig

base_model_id = "gpt2-medium"
peft_model_path = "models/backup/gpt2-medium-it-alpaca/checkpoint-55"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(base_model_id)
tokenizer.pad_token = tokenizer.eos_token

# Load PEFT config to get base model
peft_config = PeftConfig.from_pretrained(peft_model_path)

# Optional: quantized base model for low RAM inference
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype="bfloat16",  # or torch.float16 if needed
    bnb_4bit_quant_type="nf4"
)

# Load base model and attach LoRA weights
base_model = AutoModelForCausalLM.from_pretrained(
    peft_config.base_model_name_or_path,
    quantization_config=bnb_config,
    device_map="auto"  # or {"": 0}
)
7888888888888888
model = PeftModel.from_pretrained(base_model, peft_model_path)
model.eval()

model = model.merge_and_unload()  # Merges LoRA weights into base
model.save_pretrained("models/gpt2-medium-it-alpaca-merged")
tokenizer.save_pretrained("models/gpt2-medium-it-alpaca-merged")


/home/nitin/miniconda3/envs/ml/lib/python3.12/site-packages/peft/tuners/lora/bnb.py:351: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


('models/gpt2-medium-it-alpaca-merged/tokenizer_config.json',
 'models/gpt2-medium-it-alpaca-merged/special_tokens_map.json',
 'models/gpt2-medium-it-alpaca-merged/vocab.json',
 'models/gpt2-medium-it-alpaca-merged/merges.txt',
 'models/gpt2-medium-it-alpaca-merged/added_tokens.json',
 'models/gpt2-medium-it-alpaca-merged/tokenizer.json')

In [2]:
prompt = "Describe structure of a plant"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
output_base = base_model.generate(**inputs, max_new_tokens=50, do_sample=True, temperature=0.75, top_p=0.95, top_k=50)
outputs = model.generate(**inputs, max_new_tokens=50, do_sample=True, temperature=0.75, top_p=0.95, top_k=50)

print("Base model output:------------------------")
print(tokenizer.decode(output_base[0], skip_special_tokens=True))
print("\nLoRA model output:------------------------")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Base model output:------------------------
Describe structure of a plant and its function

A plant's structure is based on its structure, function, and structure. You can examine a plant's structure and function by examining the leaves of a plant, the stem of a plant, the roots, and the flower buds

LoRA model output:------------------------
Describe structure of a plant, including photosynthetic mechanisms, photosynthetic processes, and chlorophyll and chlorophyllase activity.

Describe plant morphology, including morphology, growth and development, root system, and leaf structure.

Describe plant physiological
